In [ ]:
!apt-get update
!apt-get install -y libglu1-mesa
%pip install numpy matplotlib meshio pygmsh pyvista anywidget

In [ ]:
import pygmsh
import numpy as np
import matplotlib.pyplot as plt
import json

# 設定
OUTPUT_FILE = 'input.json'

print("✅ Libraries loaded.")

In [ ]:
rules_content = """# pygmsh.occ Geometry Generation Rules (Strict)

You are a CAD/CAE expert. Output ONLY valid Python code using the `pygmsh.occ` API.
The variable `geom` is already initialized.

### 1. Mandatory API Signatures:
- Rectangle: `rect = geom.add_rectangle([x0, y0, z0], width, height)`
- Disk (Circle): `disk = geom.add_disk([xc, yc, zc], radius)`
- Boolean Union: `union = geom.boolean_union([entity1, entity2, ...])`
- Boolean Difference: `diff = geom.boolean_difference(target_entity, tool_entity)`

### 2. Code Structure & Formatting (ULTRA STRICT):
- **Comment Every Line**: Every single line of code MUST be preceded or followed by a comment on its own line or at the end of the line.
- **Mandatory Newlines**: You MUST use a hard newline after every single statement. Do NOT put multiple statements on one line.
- **Example of Required Style**:
  W = 500.0  # Define width
  H = 200.0  # Define height
  plate = geom.add_rectangle([0.0, 0.0, 0.0], W, H)  # Create base plate
- **NO Semicolons**: Semicolons are strictly forbidden.
- **Float Only**: Use float values for all dimensions (e.g., 50.0).

完全な独立行の徹底: 1つの行（Line）には、必ず1つのステートメント（代入または関数呼び出し）のみを記述すること 。


一括代入の禁止: W, H = 300.0, 100.0 のような複数変数の同時定義を禁止し、必ず W = 300.0 と H = 100.0 に分割して記述すること 。


物理的距離の確保: すべての有効なコード行の間に、必ず1行以上の空行（Blank Line）を挿入すること 。


### 3. Output Format:
- **No Markdown**: Do NOT use code blocks (```). Start directly with Python code.
- **No Explanation**: Output ONLY the code and comments. No conversational text.
- Assign the final resulting shape to `final_shape`.

### 4. Physical Groups:
- **Surface**: Always apply `geom.add_physical(final_shape, "SURFACE")`.
- **Boundaries**: For boundaries, create a thin rectangle (0.1 thickness) and label it (e.g., "FIXED_SUPPORT").

### 5. Mesh Stability:
- Avoid overlapping edges.
- Ensure all holes are within the plate boundaries.

### 6.
Extra Spacing: You MUST insert an additional empty newline between logically distinct blocks or statements to enhance readability. (論理的なブロックや文の間に、可読性を高めるための空行を必ず挿入すること。)
"""

# ファイルとして保存
with open("pygmsh_rules.txt", "w", encoding="utf-8") as f:
    f.write(rules_content)

print("✅ 'pygmsh_rules.txt' has been created successfully.")

In [ ]:
def generate_mesh_from_llm(recipe_code, mesh_size=10.0):
    """
    LLMが生成した形状レシピ（Pythonコード文字列）を実行し、
    pygmshを使用してメッシュを生成する関数。
    """
    try:
        with pygmsh.occ.Geometry() as geom:
            # メッシュサイズのグローバル設定
            geom.characteristic_length_min = mesh_size
            geom.characteristic_length_max = mesh_size

            # LLMが生成したコードを実行
            # 第2引数に、exec内で使用可能にする変数を辞書で渡します
            exec_globals = {"geom": geom}
            exec(recipe_code, exec_globals)

            # メッシュ生成
            mesh = geom.generate_mesh()
            return mesh

    except Exception as e:
        print(f"❌ Error during mesh generation: {e}")
        return None

User Prompt: 添付した pygmsh_rules.txt のルールを厳守して、以下の形状を作成するPythonコードを出力してください。

要望： 縦300、横800の板の左端を固定したい。中央に三角形のような配置で3つの小さな穴（半径10）を開け、さらに右端の上下の角に小さな切り欠きを作って。

In [ ]:
LLM_GEOMETRY_RECIPE = """
W = 800.0  # Plate width
H = 400.0  # Plate height
hole_r = 30.0  # Radius of central hole
notch_w = 60.0  # Width of right-side notch
notch_h = 40.0  # Height of right-side notch
plate = geom.add_rectangle([0.0, 0.0, 0.0], W, H)  # Base plate
hole = geom.add_disk([W / 2.0, H / 2.0, 0.0], hole_r)  # Central circular hole
notch = geom.add_rectangle([W - notch_w, H / 2.0 - notch_h / 2.0, 0.0], notch_w, notch_h)  # Right-side notch
cutouts = geom.boolean_union([hole, notch])  # Combine hole and notch
final_shape = geom.boolean_difference(plate, cutouts)  # Subtract cutouts from plate
geom.add_physical(final_shape, "SURFACE")  # Tag final surface
left_edge = geom.add_rectangle([0.0, 0.0, 0.0], 0.1, H)  # Thin left boundary for fixing
geom.add_physical(left_edge, "FIXED_LEFT")  # Tag fixed left boundar


"""

def generate_mesh_from_llm(recipe_code, mesh_size=5.0):
    import pygmsh
    try:
        # 文字列から前後の空白やマークダウンのゴミを取り除く
        clean_code = recipe_code.strip().replace("```python", "").replace("```", "")

        with pygmsh.occ.Geometry() as geom:
            geom.characteristic_length_min = mesh_size
            geom.characteristic_length_max = mesh_size

            # 実行環境にgeomを渡す
            local_vars = {"geom": geom}
            exec(clean_code, {}, local_vars)

            mesh = geom.generate_mesh()
            return mesh
    except Exception as e:
        print(f"❌ Mesh generation failed: {e}")
        return None

In [ ]:
# --- この1行を追加 ---
mesh = generate_mesh_from_llm(LLM_GEOMETRY_RECIPE, mesh_size=10.0)

In [ ]:
if mesh is not None:
    points = mesh.points
    triangles = mesh.cells_dict["triangle"]

    plt.figure(figsize=(10, 4))
    plt.triplot(points[:, 0], points[:, 1], triangles, lw=0.5, color='k')
    plt.title(f"Mesh Preview ({len(points)} nodes)")
    plt.axis('equal')
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("⚠️ No mesh to preview.")

In [ ]:
# --- 5. 保存（形状データのみ） ---
if mesh is not None:
    nodes = mesh.points[:, :2]
    elements = mesh.cells_dict["triangle"]

    # 物理条件はここでは決めず、枠だけ作っておく
    input_data = {
        "nodes": nodes.tolist(),
        "elements": elements.tolist(),
        "material": {
            "E": 210000.0,
            "thickness": 10.0
        },
        "conditions": {
            # 🌟 ここでは空にしておく（Pre側で判定させる）
            "fixed_nodes_x": [],
            "load_nodes": [],
            "total_force": 1000.0
        }
    }

    with open(OUTPUT_FILE, 'w') as f:
        json.dump(input_data, f, indent=4)

    print(f"🎉 Saved '{OUTPUT_FILE}' (Geometry only)")
    print(f"   Nodes: {len(nodes)}, Elements: {len(elements)}")
    print("   👉 Open the 'Pre' notebook to set boundary conditions.")

else:
    print("⚠️ Save skipped.")

In [ ]:
%pip install --upgrade meshio

In [ ]:
# --- 6. メッシュのエクスポート (修正版) ---
GMSH_OUTPUT_FILE = 'model_mesh.msh'

if mesh is not None:
    try:
        import meshio

        # 2D解析に必要な 'triangle' 要素のみを抽出して新しいメッシュオブジェクトを作成
        # これにより、不要な点要素や線要素による dim_tags エラーを回避します
        cells = [("triangle", mesh.cells_dict["triangle"])]

        out_mesh = meshio.Mesh(
            points=mesh.points,
            cells=cells,
            # 必要に応じて cell_data も引き継げますが、まずは最小構成で
        )

        # GMSH 2.2 形式は互換性が非常に高く、多くのソフトで UNV 同様に扱えます
        out_mesh.write(GMSH_OUTPUT_FILE, file_format="gmsh22")

        print(f"🎉 Exported triangle mesh to '{GMSH_OUTPUT_FILE}' (GMSH 2.2 format)")

    except Exception as e:
        print(f"❌ Export failed: {e}")
else:
    print("⚠️ Export skipped.")